# architecture

Template for **one architecture, many parameter configs**. To use it:
1. copy `notebooks/_template/` to `notebooks/<member>/` (once per member);
2. copy `architecture/` to `<arch>/`, named after the architecture (e.g. `resnet18/`, `convnext_tiny/`), and set `MEMBER` and `MODEL` below.

One notebook = one architecture. `TOPIC` defaults to the architecture name, so every run lands in `runs/<member>/<arch>/<run_name>/` (`config.json`, `metrics.json`, `history.csv`, `best.pt`, ...).

Cells: **1** parameter grid → **2** train every config with the shared `fit()` loop (cached: finished runs are reused) → **3** compare (table + curves) → **conclusion** → **4** log the quality metrics of each variant to MLflow.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))  # makes `app` importable; `iml` is installed by uv

from iml.config import ExperimentConfig
from iml.cli import check_run_data, data_fingerprint, load_splits, record_run_data
from iml.runs import run_dir, load_run
from iml.train import fit
from iml.cli import summary_table, plot_curves
from iml.cli import expand_grid
from iml.tracking import Tracker, notebook_run

MEMBER = "<member>"
MODEL = "resnet18"  # one of iml.config.MODEL_NAMES
TOPIC = MODEL  # runs land in runs/<member>/<arch>/<run_name>/
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://localhost:5000")  # cell 4 logs here (fit() itself does not)
DATA = os.environ.get("IML_DATA")  # dataset folder (DATA_CONTRACT.md v3, .npy clips): set $IML_DATA or a path

## 1. Parameter grid

The architecture (`MODEL`) is fixed; only parameters vary. Two ways to list the configs:
- `GRID`: `{field: [values]}`, expanded as a cartesian product (`expand_grid`), e.g. 2 × 2 × 2 = 8 runs below;
- `EXPLICIT`: hand-written lists of `key=value` overrides on top of `BASE`, one list per run. When `EXPLICIT` is non-empty it is used **instead of** `GRID`.

`variant(cfg)` names a config by the fields that differ from the defaults.

In [ ]:
BASE = ExperimentConfig(model=MODEL, member=MEMBER, topic=TOPIC, epochs=20)
GRID = {"lr": [1e-3, 3e-4], "dropout": [0.0, 0.2], "pretrained": [True, False]}
EXPLICIT: list[list[str]] = []  # e.g. [["lr=1e-3"], ["lr=3e-4", "weight_decay=0.05"]]

CONFIGS = [BASE.with_overrides(*o) for o in EXPLICIT] or expand_grid(BASE, GRID)


def variant(cfg: ExperimentConfig) -> str:
    return ",".join(f"{k}={v}" for k, v in sorted(cfg.diff_from_default().items())) or "baseline"


import pandas as pd

pd.set_option("display.width", 200)
pd.DataFrame({"run_name": [c.run_name() for c in CONFIGS], "variant": [variant(c) for c in CONFIGS]})

## 2. Train (cached)

One run per config. A finished run (`metrics.json` exists) is loaded instead of retrained, but only if it was trained on the same dataset (`check_run_data` compares the data fingerprint).

In [ ]:
FP = data_fingerprint(DATA)  # a finished run is reused only if it was trained on this dataset

records = []
for cfg in CONFIGS:
    out = run_dir(cfg)
    check_run_data(out, FP, overwrite=False)
    if (out / "metrics.json").exists():
        print("cached:", out.name)
        records.append(load_run(out))
    else:
        # tracker disabled: cell 4 logs one MLflow run per variant, so fit() must not log a second one
        records.append(fit(cfg, load_splits(cfg, data_dir=DATA), tracker=Tracker(enabled=False)))
        record_run_data(out, FP)

## 3. Compare: table and curves

In [ ]:
table = summary_table(records)
table

In [ ]:
from IPython.display import Image, display

out = run_dir(CONFIGS[0]).parent / "comparison"  # runs/<member>/<arch>/comparison/ (local only)
out.mkdir(parents=True, exist_ok=True)
plot_curves(records, out)
table.to_csv(out / "table.csv", index=False)
display(Image(out / "loss.png"), Image(out / "val_metrics.png"))

### Conclusion

_Which parameters matter for this architecture? Which config goes forward? (write it here so the report can quote it; outputs are stripped on commit)_

## 4. MLflow (quality metrics only)

One MLflow run per variant in experiment `<member>/<topic>`. **Rule:** MLflow receives only classification-quality metrics — accuracy, FAR, FRR, macro F1, EER (as `val_*` / `test_*`). Params are the model plus the fields that differ from the defaults; the tag `variant` names the config. No figures, tables, artifacts or dicts (`log_artifact` / `log_dict` are not used). Without a reachable server this only warns.

In [ ]:
import re

QUALITY = re.compile(r"^(val|test)_(accuracy|far|frr|eer|macro_f1|f1_macro)$")


def _param(v):
    return str(v) if isinstance(v, (list, tuple, dict)) else v


rows = table.set_index("run").to_dict("index")
for cfg, r in zip(CONFIGS, records):
    row = rows[r.name]
    metrics = {k: float(v) for k, v in row.items() if QUALITY.match(k) and pd.notna(v)}
    params = {"model": MODEL, **{k: _param(v) for k, v in cfg.diff_from_default().items()}}
    tags = {"variant": variant(cfg)}
    with notebook_run(member=MEMBER, topic=TOPIC, run_name=r.name, params=params, tags=tags) as run:
        run.log_metrics(metrics)